In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
from langchain.chat_models import init_chat_model
from langchain_core.rate_limiters import InMemoryRateLimiter
import re
import math
import asyncio
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

## Config

In [2]:
# Nothing works without an API key
load_dotenv()

if not os.environ["OPENAI_API_KEY"]:
    print("Please add your OpenAI API key to .env to proceed.")

In [3]:
# Tracking model costs and rate limits
MODELS = {
    "openai:gpt-5-mini": {
        "model_id": "openai:gpt-5-mini",
        "token_cost_input": 0.25 / 10 ** 6,
        "token_cost_output": 2.00 / 10 ** 6,
        "tokens_per_min": 5 * 10 ** 5,
        "tokens_per_day": 5 * 10 ** 6,
        "requests_per_min": 500,
    },
    "openai:gpt-5-nano" : {
        "model_id": "openai:gpt-5-nano",
        "token_cost_input": 0.05 / 10 ** 6,
        "token_cost_output": 0.40 / 10 ** 6,
        "tokens_per_min": 2 * 10 ** 5,
        "tokens_per_day": 2 * 10 ** 6,
        "requests_per_min": 500,
    },
}

In [4]:
# Setting constants for later use
DATA_PATH = "data/raw"
TRAIN_FILE = "train.csv"
TEST_FILE = "test.csv"
VAL_FILE = "val.csv"
TOKEN_FACTOR = 1.3
PROMPT_TEMPLATE = "What is the sentiment? Respond only with 1 for Positive or 0 for Negative):"

## Loading the data

In [5]:
# Only using test data since we are not training the LLM
test_filepath = Path(DATA_PATH) / TEST_FILE
test_df = pd.read_csv(test_filepath)

In [6]:
test_df.head()

,Text,Label
0,I am always looking for a quick and easy but n...,1
1,"We had been giving our dog the Liver Biscotti,...",1
2,It has a very foul aftertaste. Taste at first ...,0
3,I love SoBe Fruit Punch. My neighborhood Tops ...,1
4,"When I lived in TN, I was able to find this pu...",1


In [7]:
test_df["text_length"] = test_df["Text"].str.len()

In [8]:
test_df.describe()

,Label,text_length
count,5000.000000,5000.000000
mean,0.842600,419.860600
std,0.364214,411.017433
min,0.000000,32.000000
25%,1.000000,175.000000
50%,1.000000,298.000000
75%,1.000000,513.000000
max,1.000000,5790.000000


In [9]:
# Dataset is imbalanced at 85/15 split positive/negative
test_df["Label"].value_counts()

Label
1    4213
0     787
Name: count, dtype: int64

In [10]:
# How many tokens are we about to send through this thing?
num_samples = len(test_df)
prompt_len = len(PROMPT_TEMPLATE)
mean_num_words = float(test_df.describe().loc["mean"]["text_length"])
estimated_num_tokens = num_samples * (mean_num_words + prompt_len) * TOKEN_FACTOR
print(f"Estimated number of tokens for test set: {estimated_num_tokens:.3f}")

Estimated number of tokens for test set: 3216593.900


In [11]:
def evaluate_llm_cost(model: dict) -> None:
    """Given a model and its token costs, estimates total cost for the task."""
    print(f"Estimating costs for using model:\n\t{model["model_id"]}\n")
    estimated_input_cost = estimated_num_tokens * model["token_cost_input"]
    print(f"Estimated cost of input tokens: {estimated_input_cost}")
    
    estimated_output_cost = num_samples * model["token_cost_output"]
    print(f"Estimated cost of output tokens: {estimated_output_cost}")
    
    estimated_total_cost = estimated_input_cost + estimated_output_cost
    print(f"Estimated total cost of experiment: {estimated_total_cost}\n")

In [12]:
for model_id in MODELS:
    evaluate_llm_cost(MODELS[model_id])

Estimating costs for using model:
	openai:gpt-5-mini

Estimated cost of input tokens: 0.804148475
Estimated cost of output tokens: 0.01
Estimated total cost of experiment: 0.814148475

Estimating costs for using model:
	openai:gpt-5-nano

Estimated cost of input tokens: 0.16082969500000002
Estimated cost of output tokens: 0.002
Estimated total cost of experiment: 0.16282969500000002



These models are much more cost efficient than I though they would be, which is impressive. However, the real challenge appears to be abiding by the rate limits. For gpt-5-nano the total tokens per day is 2M, so it would take two days just for a single full pass on the data. For that reason, I switched to the more "expensive" model gpt-5-mini.

## Setting up the model

In [13]:
# Set model choice from MODELS
MODEL_ID = "openai:gpt-5-mini"
TEMP = 0 # More consistent LLM outputs

# Get rate limit info
REQUESTS_PER_MIN = MODELS[MODEL_ID]["requests_per_min"]
TOKENS_PER_MIN = MODELS[MODEL_ID]["tokens_per_min"]
TOKENS_PER_DAY = MODELS[MODEL_ID]["tokens_per_day"]
TARGET_REQUESTS_PER_MIN = REQUESTS_PER_MIN * 0.75 # Hit 75% of max
TARGET_REQUESTS_PER_SEC = TARGET_REQUESTS_PER_MIN / 60
MAX_RETRIES = 3
STOP_AFTER_ATTEMPT = 3

# Set generation params
BATCH_SIZE = 50 # How many reviews sent per parallel worker
MAX_CONCURRENCY = 10 # How many max parallel workers

In [14]:
# Set rate limiting as appropriate
rate_limiter = InMemoryRateLimiter(
    requests_per_second = TARGET_REQUESTS_PER_SEC,
    check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
    max_bucket_size=10,  # Controls the maximum burst size.
)

# Initialize rate-limited model
model = init_chat_model(
    MODEL_ID,
    rate_limiter = rate_limiter,
    max_retries = MAX_RETRIES,
    temperature = TEMP
).with_retry(stop_after_attempt=STOP_AFTER_ATTEMPT)

## Generating responses

In [15]:
def build_review_prompt(review: str) -> str:
    prompt = f"{PROMPT_TEMPLATE}\n{review}"
    return prompt

In [16]:
def parse_prediction(prediction: str) -> int:
    """Attempts to retrieve an integer prediction from the LLM prediction response."""
    if prediction is None:
        return -1
    match = re.search(r"-?\d+", str(prediction).strip())
    return int(match.group()) if match else -1

In [17]:
async def generate_llm_predictions(
    reviews: list[str],
    batch_size: int = 50,
    max_concurrency: int = 10
) -> (int, dict):
    """Generates predictions on the given reviews using the preset model."""
    review_prompts = [build_review_prompt(review) for review in reviews]
    predictions = []

    # Process reviews in batches up to batch_size
    for index in range(0, len(review_prompts), batch_size):
        review_prompts_chunk = review_prompts[index:index+batch_size]

        # Generate responses in batch up to max_concurrency in parallel
        responses = await model.abatch(
            review_prompts_chunk,
            config = {"max_concurrency": max_concurrency},
            return_exceptions = True,
        )

        # Process reponses
        for res in responses:
            # API calls might fail for any number of reasons
            if isinstance(res, Exception):
                print(f"Request failed: {res}")
                predictions.append(-1)
                continue

            # Dealing with an LLM here, so it might not stick to the script...
            prediction = parse_prediction(res.content)
            if prediction == -1:
                print(f"Could not parse prediction from response: {res.content}")
            predictions.append(prediction)

    return predictions

In [18]:
# Generate and track predictions!
batch_reviews = list(test_df["Text"])
predictions = await generate_llm_predictions(
    batch_reviews,
    batch_size = BATCH_SIZE,
    max_concurrency = MAX_CONCURRENCY
)

In [19]:
test_df["model_prediction"] = predictions

In [20]:
# Save results to file, don't want to wait on that again...
results = Path("results")
results.mkdir(parents=True, exist_ok=True)
test_df.to_csv(f"results/{MODEL_ID.replace(":","_")}.csv")

## Evaluating the model responses

In [21]:
print(f"Evaluating model:\n\t{MODEL_ID}")
test_accuracy = accuracy_score(test_df["Label"], test_df["model_prediction"])
print(f"Test Accuracy:\n\t{test_accuracy}")

test_classification_report = classification_report(test_df["Label"], test_df["model_prediction"])
print(f"Classification Report:\n{test_classification_report}")


Evaluating model:
	openai:gpt-5-mini
Test Accuracy:
	0.9756
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.97      0.93       787
           1       0.99      0.98      0.99      4213

    accuracy                           0.98      5000
   macro avg       0.94      0.97      0.96      5000
weighted avg       0.98      0.98      0.98      5000



## Summary

I checked the OpenAI platform after this run and the total number of tokens (input & output) came to 605,894 tokens. This could be explained by their token caching, but I did not realize it would be this efficient as compared to my estimate.

The total spend came out to `$ 0.73`, which was also below the estimmated cost of `$ 0.81`. This is also likely attributable to the token caching that OpenAI performs.

Overall, cost proved not to be much of a limiting factor as was anticpated. Rather, the difficulty came in abiding by the rate limits OpenAI sets for their models. This drove the choice in using gpt-5-mini. LangChain also greatly reduced the technical barrier for adhering to the rate limits.

## References

- [LangChain init_chat_model](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model)
- [OpenAI API Pricing](https://openai.com/api/pricing/)
- [Rate-limiting](https://docs.langchain.com/langsmith/rate-limiting)
- [BaseChatModel](https://reference.langchain.com/python/langchain-core/language_models/chat_models/BaseChatModel)